# 9 Refit Selected Tuned Models on Full Train-Validation Set

## 9.0  Purpose and fixed rules

Fixed rules:
- No new tuning.
- No feature redesign.
- No model selection based on train scores.
- No final test predictions.
- No final test metrics.
- Final test sets may be recreated only for split consistency checks.
- Use carry-forward parameters, not raw GridSearchCV best params.
- Fit full sklearn Pipelines: preprocessing + model.
- Save fitted pipelines, metadata, fitted feature names, package versions, and sanity summaries.

## 9.1  Imports, constants, file paths

In [2]:
# imports
from pathlib import Path
import json
import time
import platform

import joblib
import numpy as np
import pandas as pd

import sklearn

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss
)

In [4]:
# constants
RANDOM_STATE = 42
N_JOBS = 1

GROUP_COL = "mailing_id"

OPEN_TARGET = "open"
CLICK_TARGET = "click"

TEST_CAMPAIGN_FRACTION = 0.20

In [5]:
# file paths
THESIS_DIR = Path.home() / "Documents" / "Thesis"

NOTEBOOK_DIR = THESIS_DIR / "Notebooks"
OUTPUT_DIR = THESIS_DIR / "Outputs"

DATA_DIR = THESIS_DIR / "Data"
PROCESSED_DATA_DIR = DATA_DIR / "Processed Data"

OPEN_MODEL_PATH = PROCESSED_DATA_DIR / "df_open_model_v1.parquet"
CLICK_MODEL_PATH = PROCESSED_DATA_DIR / "df_click_model_v1.parquet"

OPEN_PARAMS_PATH = OUTPUT_DIR / "7_open_carry_forward_params_v1.json"
CLICK_PARAMS_PATH = OUTPUT_DIR / "8_click_carry_forward_params_v1.json"

print("Thesis directory:", THESIS_DIR)
print("Notebook directory exists:", NOTEBOOK_DIR.exists())
print("Output directory exists:", OUTPUT_DIR.exists())
print("Processed data directory exists:", PROCESSED_DATA_DIR.exists())

Thesis directory: /Users/khanhhien/Documents/Thesis
Notebook directory exists: True
Output directory exists: True
Processed data directory exists: True


In [7]:
critical_files = {
    "Open dataset": OPEN_MODEL_PATH,
    "Click dataset": CLICK_MODEL_PATH,
    "Open carry-forward params": OPEN_PARAMS_PATH,
    "Click carry-forward params": CLICK_PARAMS_PATH,
}

for name, path in critical_files.items():
    print(f"{name}: {path.exists()}")

missing_files = [name for name, path in critical_files.items() if not path.exists()]

assert len(missing_files) == 0, f"Missing files: {missing_files}"

Open dataset: True
Click dataset: True
Open carry-forward params: True
Click carry-forward params: True


## 9.2  Load open/click datasets

In [8]:
# load datasets
df_open = pd.read_parquet(OPEN_MODEL_PATH)
df_click = pd.read_parquet(CLICK_MODEL_PATH)

print("Open dataset loaded:", df_open.shape)
print("Click dataset loaded:", df_click.shape)

Open dataset loaded: (1057544, 38)
Click dataset loaded: (1104242, 38)


In [9]:
# basic column checks
required_open_cols = [OPEN_TARGET, GROUP_COL]
required_click_cols = [CLICK_TARGET, GROUP_COL]

missing_open_cols = [col for col in required_open_cols if col not in df_open.columns]
missing_click_cols = [col for col in required_click_cols if col not in df_click.columns]

assert len(missing_open_cols) == 0, f"Missing open columns: {missing_open_cols}"
assert len(missing_click_cols) == 0, f"Missing click columns: {missing_click_cols}"

In [10]:
# target and campaign checks
print("Open target missing values:", df_open[OPEN_TARGET].isna().sum())
print("Click target missing values:", df_click[CLICK_TARGET].isna().sum())

assert df_open[OPEN_TARGET].isna().sum() == 0, "Open target has missing values."
assert df_click[CLICK_TARGET].isna().sum() == 0, "Click target has missing values."

print("\nOpen target distribution:")
print(df_open[OPEN_TARGET].value_counts(normalize=True).sort_index())

print("\nClick target distribution:")
print(df_click[CLICK_TARGET].value_counts(normalize=True).sort_index())

print("\nOpen unique campaigns:", df_open[GROUP_COL].nunique())
print("Click unique campaigns:", df_click[GROUP_COL].nunique())

print("\nOpen campaign range:", df_open[GROUP_COL].min(), "to", df_open[GROUP_COL].max())
print("Click campaign range:", df_click[GROUP_COL].min(), "to", df_click[GROUP_COL].max())

Open target missing values: 0
Click target missing values: 0

Open target distribution:
open
0.0    0.46382
1.0    0.53618
Name: proportion, dtype: float64

Click target distribution:
click
0.0    0.998918
1.0    0.001082
Name: proportion, dtype: float64

Open unique campaigns: 274
Click unique campaigns: 274

Open campaign range: 653 to 1418
Click campaign range: 653 to 1418


## 9.3  Recreate chronological trainval/test split

In [11]:
# recreate campaign split
campaign_ids = sorted(df_open[GROUP_COL].unique())

n_campaigns = len(campaign_ids)
n_trainval_campaigns = int(n_campaigns * (1 - TEST_CAMPAIGN_FRACTION))

trainval_campaign_ids = campaign_ids[:n_trainval_campaigns]
test_campaign_ids = campaign_ids[n_trainval_campaigns:]

print("Total campaigns:", n_campaigns)
print("Trainval campaigns:", len(trainval_campaign_ids))
print("Test campaigns:", len(test_campaign_ids))

Total campaigns: 274
Trainval campaigns: 219
Test campaigns: 55


In [12]:
# campaign checks
print("Trainval campaign range:")
print(min(trainval_campaign_ids), "to", max(trainval_campaign_ids))

print("\nTest campaign range:")
print(min(test_campaign_ids), "to", max(test_campaign_ids))

overlap = set(trainval_campaign_ids).intersection(test_campaign_ids)

print("\nCampaign overlap:", len(overlap))

assert len(overlap) == 0

Trainval campaign range:
653 to 1252

Test campaign range:
1257 to 1418

Campaign overlap: 0


## 9.4  Validate split consistency

In [13]:
# create trainval/test datasets
open_trainval_df = df_open[
    df_open[GROUP_COL].isin(trainval_campaign_ids)
].copy()

open_test_df = df_open[
    df_open[GROUP_COL].isin(test_campaign_ids)
].copy()

click_trainval_df = df_click[
    df_click[GROUP_COL].isin(trainval_campaign_ids)
].copy()

click_test_df = df_click[
    df_click[GROUP_COL].isin(test_campaign_ids)
].copy()

In [14]:
# dataset shapes
print("Open trainval:", open_trainval_df.shape)
print("Open test:", open_test_df.shape)

print()

print("Click trainval:", click_trainval_df.shape)
print("Click test:", click_test_df.shape)

Open trainval: (781826, 38)
Open test: (275718, 38)

Click trainval: (814269, 38)
Click test: (289973, 38)


In [15]:
# camapgin count validation
print(
    "Open trainval campaigns:",
    open_trainval_df[GROUP_COL].nunique()
)

print(
    "Open test campaigns:",
    open_test_df[GROUP_COL].nunique()
)

print()

print(
    "Click trainval campaigns:",
    click_trainval_df[GROUP_COL].nunique()
)

print(
    "Click test campaigns:",
    click_test_df[GROUP_COL].nunique()
)

Open trainval campaigns: 219
Open test campaigns: 55

Click trainval campaigns: 219
Click test campaigns: 55


## 9.5  Define open/click C-expanded feature sets

In [16]:
# open feature set C
open_feature_types_C = {
    "numeric": [
        "age_clean",
        "n_interests",
        "subject_length",
        "preheader_length",
        "prior_email_count",
        "prior_open_count",
        "prior_click_count",
        "historical_open_rate",
        "historical_click_rate",
    ],
    "categorical": [
        "gender",
        "age_group",
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "historical_open_bucket",
        "prior_email_frequency_bucket",
    ],
    "binary": [
        "interest_topic_match",
        "has_campaign_metadata",
        "subject_missing",
        "subject_contains_number",
        "subject_contains_promotion",
        "subject_contains_exclamation",
        "preheader_missing",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "preheader_contains_exclamation",
        "had_prior_click",
        "is_first_email",
    ],
}

open_feature_set_C_expanded = (
    open_feature_types_C["numeric"]
    + open_feature_types_C["categorical"]
    + open_feature_types_C["binary"]
)

print("Open C-expanded feature count:", len(open_feature_set_C_expanded))

Open C-expanded feature count: 28


In [17]:
# click feature set C
click_feature_types_C = {
    "numeric": [
        "age_clean",
        "n_interests",
        "subject_length",
        "preheader_length",
        "prior_email_count",
        "prior_open_count",
        "prior_click_count",
        "historical_open_rate",
        "historical_click_rate",
    ],
    "categorical": [
        "gender",
        "age_group",
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "historical_open_bucket",
        "prior_email_frequency_bucket",
    ],
    "binary": [
        "interest_topic_match",
        "has_campaign_metadata",
        "subject_missing",
        "subject_contains_number",
        "subject_contains_promotion",
        "subject_contains_exclamation",
        "preheader_missing",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "preheader_contains_exclamation",
        "had_prior_click",
        "is_first_email",
    ],
}

click_feature_set_C_expanded = (
    click_feature_types_C["numeric"]
    + click_feature_types_C["categorical"]
    + click_feature_types_C["binary"]
)

print("Click C-expanded feature count:", len(click_feature_set_C_expanded))

Click C-expanded feature count: 28


In [18]:
# feature set validation
def validate_feature_set(feature_set, feature_types, df, target_name, forbidden_cols):
    duplicated_features = pd.Series(feature_set)[pd.Series(feature_set).duplicated()].tolist()
    missing_features = [col for col in feature_set if col not in df.columns]
    forbidden_found = [col for col in forbidden_cols if col in feature_set]
    
    typed_features = (
        feature_types["numeric"]
        + feature_types["categorical"]
        + feature_types["binary"]
    )
    
    untyped_or_mismatch = sorted(set(feature_set).symmetric_difference(set(typed_features)))
    
    print(f"\n{target_name} feature validation")
    print("Feature count:", len(feature_set))
    print("Duplicated features:", duplicated_features)
    print("Missing features:", missing_features)
    print("Forbidden columns found:", forbidden_found)
    print("Feature/type mismatch:", untyped_or_mismatch)
    
    assert len(feature_set) == 28, f"{target_name}: C-expanded should have 28 features."
    assert len(duplicated_features) == 0, f"{target_name}: duplicated features found."
    assert len(missing_features) == 0, f"{target_name}: missing features found."
    assert len(forbidden_found) == 0, f"{target_name}: forbidden columns found."
    assert len(untyped_or_mismatch) == 0, f"{target_name}: feature/type mismatch."

validate_feature_set(
    feature_set=open_feature_set_C_expanded,
    feature_types=open_feature_types_C,
    df=open_trainval_df,
    target_name="Open",
    forbidden_cols=["open", "click", "user_id", "mailing_id"]
)

validate_feature_set(
    feature_set=click_feature_set_C_expanded,
    feature_types=click_feature_types_C,
    df=click_trainval_df,
    target_name="Click",
    forbidden_cols=["click", "open", "user_id", "mailing_id"]
)


Open feature validation
Feature count: 28
Duplicated features: []
Missing features: []
Forbidden columns found: []
Feature/type mismatch: []

Click feature validation
Feature count: 28
Duplicated features: []
Missing features: []
Forbidden columns found: []
Feature/type mismatch: []


## 9.6  Build preprocessing function

In [20]:
# build preprocessor function
def build_preprocessor(feature_types):
    numeric_features = feature_types["numeric"]
    categorical_features = feature_types["categorical"]
    binary_features = feature_types["binary"]
    
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("scaler", StandardScaler())
    ])
    
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    
    binary_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
            ("binary", binary_pipeline, binary_features),
        ],
        remainder="drop"
    )
    
    return preprocessor

In [21]:
# preprocessors check
open_preprocessor_C = build_preprocessor(open_feature_types_C)
click_preprocessor_C = build_preprocessor(click_feature_types_C)

print("Open preprocessor created:", type(open_preprocessor_C))
print("Click preprocessor created:", type(click_preprocessor_C))

Open preprocessor created: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
Click preprocessor created: <class 'sklearn.compose._column_transformer.ColumnTransformer'>


## 9.7  Load carry-forward params

In [22]:
# carry forward params
with open(OPEN_PARAMS_PATH, "r") as f:
    open_carry_forward_params = json.load(f)

with open(CLICK_PARAMS_PATH, "r") as f:
    click_carry_forward_params = json.load(f)

print("Open carry-forward params:")
print(json.dumps(open_carry_forward_params, indent=4))

print("\nClick carry-forward params:")
print(json.dumps(click_carry_forward_params, indent=4))

Open carry-forward params:
{
    "logistic_regression__C_expanded": {
        "model__C": 0.001,
        "model__penalty": "l2",
        "model__class_weight": null
    },
    "decision_tree__C_expanded": {
        "model__max_depth": 3,
        "model__min_samples_leaf": 250
    },
    "random_forest__C_expanded": {
        "model__n_estimators": 200,
        "model__max_depth": 10,
        "model__min_samples_leaf": 250,
        "model__max_features": "sqrt",
        "model__class_weight": null
    }
}

Click carry-forward params:
{
    "logistic_regression__C_expanded": {
        "model__C": 0.1,
        "model__penalty": "l2",
        "model__class_weight": null
    },
    "random_forest__C_expanded": {
        "model__n_estimators": 200,
        "model__max_depth": 10,
        "model__min_samples_leaf": 5,
        "model__max_features": 0.5,
        "model__class_weight": null
    }
}


## 9.8  Validate parameter dictionaries

In [23]:
# validate
expected_open_keys = {
    "logistic_regression__C_expanded",
    "decision_tree__C_expanded",
    "random_forest__C_expanded",
}

expected_click_keys = {
    "logistic_regression__C_expanded",
    "random_forest__C_expanded",
}

actual_open_keys = set(open_carry_forward_params.keys())
actual_click_keys = set(click_carry_forward_params.keys())

print("Open keys correct:", actual_open_keys == expected_open_keys)
print("Click keys correct:", actual_click_keys == expected_click_keys)

print("\nOpen keys:", actual_open_keys)
print("Click keys:", actual_click_keys)

assert actual_open_keys == expected_open_keys, "Open carry-forward keys do not match expected keys."
assert actual_click_keys == expected_click_keys, "Click carry-forward keys do not match expected keys."

Open keys correct: True
Click keys correct: True

Open keys: {'decision_tree__C_expanded', 'logistic_regression__C_expanded', 'random_forest__C_expanded'}
Click keys: {'logistic_regression__C_expanded', 'random_forest__C_expanded'}


In [24]:
# click rf check
click_rf_max_features = click_carry_forward_params[
    "random_forest__C_expanded"
]["model__max_features"]

print("Click RF max_features:", click_rf_max_features)
print("Type:", type(click_rf_max_features))

assert click_rf_max_features == 0.5, "Click RF max_features should be 0.5."
assert isinstance(click_rf_max_features, float), "Click RF max_features should be float, not string."

Click RF max_features: 0.5
Type: <class 'float'>


## 9.9  Build final model registry

In [25]:
# final model registry
final_model_registry = [
    {
        "target": "open",
        "candidate_key": "logistic_regression__C_expanded",
        "model_type": "logistic_regression",
        "feature_set": open_feature_set_C_expanded,
        "feature_types": open_feature_types_C,
        "trainval_df": open_trainval_df,
        "params": open_carry_forward_params["logistic_regression__C_expanded"],
        "output_file": "9_open_refit_model_logistic_C_v1.joblib",
    },
    {
        "target": "open",
        "candidate_key": "decision_tree__C_expanded",
        "model_type": "decision_tree",
        "feature_set": open_feature_set_C_expanded,
        "feature_types": open_feature_types_C,
        "trainval_df": open_trainval_df,
        "params": open_carry_forward_params["decision_tree__C_expanded"],
        "output_file": "9_open_refit_model_decision_tree_C_v1.joblib",
    },
    {
        "target": "open",
        "candidate_key": "random_forest__C_expanded",
        "model_type": "random_forest",
        "feature_set": open_feature_set_C_expanded,
        "feature_types": open_feature_types_C,
        "trainval_df": open_trainval_df,
        "params": open_carry_forward_params["random_forest__C_expanded"],
        "output_file": "9_open_refit_model_random_forest_C_v1.joblib",
    },
    {
        "target": "click",
        "candidate_key": "logistic_regression__C_expanded",
        "model_type": "logistic_regression",
        "feature_set": click_feature_set_C_expanded,
        "feature_types": click_feature_types_C,
        "trainval_df": click_trainval_df,
        "params": click_carry_forward_params["logistic_regression__C_expanded"],
        "output_file": "9_click_refit_model_logistic_C_v1.joblib",
    },
    {
        "target": "click",
        "candidate_key": "random_forest__C_expanded",
        "model_type": "random_forest",
        "feature_set": click_feature_set_C_expanded,
        "feature_types": click_feature_types_C,
        "trainval_df": click_trainval_df,
        "params": click_carry_forward_params["random_forest__C_expanded"],
        "output_file": "9_click_refit_model_random_forest_C_v1.joblib",
    },
]

print("Number of final models:", len(final_model_registry))

Number of final models: 5


In [26]:
# registry validation
for i, model_info in enumerate(final_model_registry, start=1):

    print(f"\nModel {i}")

    print("Target:", model_info["target"])
    print("Candidate:", model_info["candidate_key"])
    print("Type:", model_info["model_type"])
    print("Features:", len(model_info["feature_set"]))
    print("Output:", model_info["output_file"])

    assert len(model_info["feature_set"]) == 28


Model 1
Target: open
Candidate: logistic_regression__C_expanded
Type: logistic_regression
Features: 28
Output: 9_open_refit_model_logistic_C_v1.joblib

Model 2
Target: open
Candidate: decision_tree__C_expanded
Type: decision_tree
Features: 28
Output: 9_open_refit_model_decision_tree_C_v1.joblib

Model 3
Target: open
Candidate: random_forest__C_expanded
Type: random_forest
Features: 28
Output: 9_open_refit_model_random_forest_C_v1.joblib

Model 4
Target: click
Candidate: logistic_regression__C_expanded
Type: logistic_regression
Features: 28
Output: 9_click_refit_model_logistic_C_v1.joblib

Model 5
Target: click
Candidate: random_forest__C_expanded
Type: random_forest
Features: 28
Output: 9_click_refit_model_random_forest_C_v1.joblib


## 9.10 Estimator factory functions

In [30]:
def build_estimator(model_type):

    if model_type == "logistic_regression":

        return LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=5000
        )

    elif model_type == "decision_tree":

        return DecisionTreeClassifier(
            random_state=RANDOM_STATE
        )

    elif model_type == "random_forest":

        return RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    else:

        raise ValueError(
            f"Unknown model_type: {model_type}"
        )

In [31]:
# smoke test
for model_type in [
    "logistic_regression",
    "decision_tree",
    "random_forest"
]:

    estimator = build_estimator(model_type)

    print(
        model_type,
        "->",
        type(estimator).__name__
    )

logistic_regression -> LogisticRegression
decision_tree -> DecisionTreeClassifier
random_forest -> RandomForestClassifier


In [32]:
# parameter application smoke test
test_spec = final_model_registry[0]

estimator = build_estimator(
    test_spec["model_type"]
)

print("Before:")

print(estimator.get_params()["C"])

estimator.set_params(
    **{
        k.replace("model__", ""): v
        for k, v in test_spec["params"].items()
    }
)

print("After:")

print(estimator.get_params()["C"])

Before:
1.0
After:
0.001


## 9.11 Build full sklearn pipeline

In [33]:
# pipeline
def build_pipeline(model_type, feature_types, params):

    preprocessor = build_preprocessor(feature_types)

    estimator = build_estimator(model_type)

    clean_params = {
        k.replace("model__", ""): v
        for k, v in params.items()
    }

    estimator.set_params(**clean_params)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    return pipeline

In [34]:
# open logistic smoke test
test_spec = final_model_registry[0]

test_pipeline = build_pipeline(
    model_type=test_spec["model_type"],
    feature_types=test_spec["feature_types"],
    params=test_spec["params"]
)

print(type(test_pipeline))

print()

print(test_pipeline.named_steps.keys())

print()

print(
    test_pipeline.named_steps["model"]
)

<class 'sklearn.pipeline.Pipeline'>

dict_keys(['preprocessor', 'model'])

LogisticRegression(C=0.001, max_iter=5000, random_state=42)


In [35]:
# pipeline test
for spec in final_model_registry:

    pipeline = build_pipeline(
        model_type=spec["model_type"],
        feature_types=spec["feature_types"],
        params=spec["params"]
    )

    print(
        spec["candidate_key"],
        "✓"
    )

logistic_regression__C_expanded ✓
decision_tree__C_expanded ✓
random_forest__C_expanded ✓
logistic_regression__C_expanded ✓
random_forest__C_expanded ✓


## 9.10 Refit Open Logistic C

In [42]:
# inputs
open_logistic_spec = final_model_registry[0]

assert open_logistic_spec["target"] == "open"
assert open_logistic_spec["candidate_key"] == "logistic_regression__C_expanded"

X_open_logistic_trainval = open_trainval_df[
    open_logistic_spec["feature_set"]
].copy()

y_open_logistic_trainval = open_trainval_df[
    OPEN_TARGET
].copy()

print("X shape:", X_open_logistic_trainval.shape)
print("y shape:", y_open_logistic_trainval.shape)

print("\nTarget distribution:")
print(y_open_logistic_trainval.value_counts(normalize=True).sort_index())

X shape: (781826, 28)
y shape: (781826,)

Target distribution:
open
0.0    0.426727
1.0    0.573273
Name: proportion, dtype: float64


In [43]:
# build pipeline
open_logistic_pipeline = build_pipeline(
    model_type=open_logistic_spec["model_type"],
    feature_types=open_logistic_spec["feature_types"],
    params=open_logistic_spec["params"]
)

print(open_logistic_pipeline.named_steps["model"])

LogisticRegression(C=0.001, max_iter=5000, random_state=42)


In [44]:
# fit
open_logistic_fit_start = time.time()

open_logistic_pipeline.fit(
    X_open_logistic_trainval,
    y_open_logistic_trainval
)

open_logistic_fit_runtime_seconds = time.time() - open_logistic_fit_start

print(f"Open Logistic C fit completed in {open_logistic_fit_runtime_seconds:.2f} seconds.")

Open Logistic C fit completed in 5.00 seconds.


In [45]:
# verify
open_logistic_model = open_logistic_pipeline.named_steps["model"]

print("Classes:", open_logistic_model.classes_)
print("Coefficient shape:", open_logistic_model.coef_.shape)
print("Intercept:", open_logistic_model.intercept_)

Classes: [0. 1.]
Coefficient shape: (1, 60)
Intercept: [-0.16419366]


## 9.11 Refit Open Decision Tree C

In [46]:
# inputs
open_tree_spec = final_model_registry[1]

assert open_tree_spec["target"] == "open"
assert open_tree_spec["candidate_key"] == "decision_tree__C_expanded"

X_open_tree_trainval = open_trainval_df[
    open_tree_spec["feature_set"]
].copy()

y_open_tree_trainval = open_trainval_df[
    OPEN_TARGET
].copy()

print("X shape:", X_open_tree_trainval.shape)
print("y shape:", y_open_tree_trainval.shape)

print("\nTarget distribution:")
print(y_open_tree_trainval.value_counts(normalize=True).sort_index())

X shape: (781826, 28)
y shape: (781826,)

Target distribution:
open
0.0    0.426727
1.0    0.573273
Name: proportion, dtype: float64


In [47]:
# build pipeline
open_tree_pipeline = build_pipeline(
    model_type=open_tree_spec["model_type"],
    feature_types=open_tree_spec["feature_types"],
    params=open_tree_spec["params"]
)

print(open_tree_pipeline.named_steps["model"])

DecisionTreeClassifier(max_depth=3, min_samples_leaf=250, random_state=42)


In [48]:
# fit
open_tree_fit_start = time.time()

open_tree_pipeline.fit(
    X_open_tree_trainval,
    y_open_tree_trainval
)

open_tree_fit_runtime_seconds = time.time() - open_tree_fit_start

print(f"Open Decision Tree C fit completed in {open_tree_fit_runtime_seconds:.2f} seconds.")

Open Decision Tree C fit completed in 3.02 seconds.


In [49]:
# verify
open_tree_model = open_tree_pipeline.named_steps["model"]

print("Classes:", open_tree_model.classes_)
print("Tree depth:", open_tree_model.get_depth())
print("Number of leaves:", open_tree_model.get_n_leaves())
print("Feature importances shape:", open_tree_model.feature_importances_.shape)

Classes: [0. 1.]
Tree depth: 3
Number of leaves: 8
Feature importances shape: (60,)


## 9.12 Refit Open Random Forest C

In [50]:
# inputs
open_rf_spec = final_model_registry[2]

assert open_rf_spec["target"] == "open"
assert open_rf_spec["candidate_key"] == "random_forest__C_expanded"

X_open_rf_trainval = open_trainval_df[
    open_rf_spec["feature_set"]
].copy()

y_open_rf_trainval = open_trainval_df[
    OPEN_TARGET
].copy()

print("X shape:", X_open_rf_trainval.shape)
print("y shape:", y_open_rf_trainval.shape)

print("\nTarget distribution:")
print(y_open_rf_trainval.value_counts(normalize=True).sort_index())

X shape: (781826, 28)
y shape: (781826,)

Target distribution:
open
0.0    0.426727
1.0    0.573273
Name: proportion, dtype: float64


In [51]:
# pipeline
open_rf_pipeline = build_pipeline(
    model_type=open_rf_spec["model_type"],
    feature_types=open_rf_spec["feature_types"],
    params=open_rf_spec["params"]
)

print(open_rf_pipeline.named_steps["model"])

RandomForestClassifier(max_depth=10, min_samples_leaf=250, n_estimators=200,
                       n_jobs=-1, random_state=42)


In [52]:
# fit
open_rf_fit_start = time.time()

open_rf_pipeline.fit(
    X_open_rf_trainval,
    y_open_rf_trainval
)

open_rf_fit_runtime_seconds = time.time() - open_rf_fit_start

print(f"Open Random Forest C fit completed in {open_rf_fit_runtime_seconds:.2f} seconds.")

Open Random Forest C fit completed in 37.00 seconds.


In [53]:
# verify
open_rf_model = open_rf_pipeline.named_steps["model"]

print("Classes:", open_rf_model.classes_)
print("Number of trees:", len(open_rf_model.estimators_))
print("Feature importances shape:", open_rf_model.feature_importances_.shape)

Classes: [0. 1.]
Number of trees: 200
Feature importances shape: (60,)


## 9.13 Refit Click Logistic C

In [54]:
# inputs
click_logistic_spec = final_model_registry[3]

assert click_logistic_spec["target"] == "click"
assert click_logistic_spec["candidate_key"] == "logistic_regression__C_expanded"

X_click_logistic_trainval = click_trainval_df[
    click_logistic_spec["feature_set"]
].copy()

y_click_logistic_trainval = click_trainval_df[
    CLICK_TARGET
].copy()

print("X shape:", X_click_logistic_trainval.shape)
print("y shape:", y_click_logistic_trainval.shape)

print("\nTarget distribution:")
print(y_click_logistic_trainval.value_counts(normalize=True).sort_index())

print("\nPositive click count:")
print(int(y_click_logistic_trainval.sum()))

X shape: (814269, 28)
y shape: (814269,)

Target distribution:
click
0.0    0.998772
1.0    0.001228
Name: proportion, dtype: float64

Positive click count:
1000


In [55]:
# pipeline
click_logistic_pipeline = build_pipeline(
    model_type=click_logistic_spec["model_type"],
    feature_types=click_logistic_spec["feature_types"],
    params=click_logistic_spec["params"]
)

print(click_logistic_pipeline.named_steps["model"])

LogisticRegression(C=0.1, max_iter=5000, random_state=42)


In [56]:
# fit
click_logistic_fit_start = time.time()

click_logistic_pipeline.fit(
    X_click_logistic_trainval,
    y_click_logistic_trainval
)

click_logistic_fit_runtime_seconds = time.time() - click_logistic_fit_start

print(f"Click Logistic C fit completed in {click_logistic_fit_runtime_seconds:.2f} seconds.")

Click Logistic C fit completed in 4.85 seconds.


In [57]:
# verify
click_logistic_model = click_logistic_pipeline.named_steps["model"]

print("Classes:", click_logistic_model.classes_)
print("Coefficient shape:", click_logistic_model.coef_.shape)
print("Intercept:", click_logistic_model.intercept_)

Classes: [0. 1.]
Coefficient shape: (1, 60)
Intercept: [-2.83395678]


## 9.14 Refit Click Random Forest C

In [58]:
# inputs
click_rf_spec = final_model_registry[4]

assert click_rf_spec["target"] == "click"
assert click_rf_spec["candidate_key"] == "random_forest__C_expanded"

X_click_rf_trainval = click_trainval_df[
    click_rf_spec["feature_set"]
].copy()

y_click_rf_trainval = click_trainval_df[
    CLICK_TARGET
].copy()

print("X shape:", X_click_rf_trainval.shape)
print("y shape:", y_click_rf_trainval.shape)

print("\nTarget distribution:")
print(y_click_rf_trainval.value_counts(normalize=True).sort_index())

print("\nPositive click count:")
print(int(y_click_rf_trainval.sum()))

X shape: (814269, 28)
y shape: (814269,)

Target distribution:
click
0.0    0.998772
1.0    0.001228
Name: proportion, dtype: float64

Positive click count:
1000


In [59]:
# pipeline
click_rf_pipeline = build_pipeline(
    model_type=click_rf_spec["model_type"],
    feature_types=click_rf_spec["feature_types"],
    params=click_rf_spec["params"]
)

print(click_rf_pipeline.named_steps["model"])

RandomForestClassifier(max_depth=10, max_features=0.5, min_samples_leaf=5,
                       n_estimators=200, n_jobs=-1, random_state=42)


In [60]:
# fit
click_rf_fit_start = time.time()

click_rf_pipeline.fit(
    X_click_rf_trainval,
    y_click_rf_trainval
)

click_rf_fit_runtime_seconds = time.time() - click_rf_fit_start

print(f"Click Random Forest C fit completed in {click_rf_fit_runtime_seconds:.2f} seconds.")

Click Random Forest C fit completed in 131.30 seconds.


In [61]:
# verify
click_rf_model = click_rf_pipeline.named_steps["model"]

print("Classes:", click_rf_model.classes_)
print("Number of trees:", len(click_rf_model.estimators_))
print("Feature importances shape:", click_rf_model.feature_importances_.shape)

Classes: [0. 1.]
Number of trees: 200
Feature importances shape: (60,)


## 9.15 Extract fitted feature names

In [63]:
fitted_feature_names = {
    "open__logistic_regression__C_expanded": (
        open_logistic_pipeline
        .named_steps["preprocessor"]
        .get_feature_names_out()
        .tolist()
    ),
    "open__decision_tree__C_expanded": (
        open_tree_pipeline
        .named_steps["preprocessor"]
        .get_feature_names_out()
        .tolist()
    ),
    "open__random_forest__C_expanded": (
        open_rf_pipeline
        .named_steps["preprocessor"]
        .get_feature_names_out()
        .tolist()
    ),
    "click__logistic_regression__C_expanded": (
        click_logistic_pipeline
        .named_steps["preprocessor"]
        .get_feature_names_out()
        .tolist()
    ),
    "click__random_forest__C_expanded": (
        click_rf_pipeline
        .named_steps["preprocessor"]
        .get_feature_names_out()
        .tolist()
    ),
}

for model_key, names in fitted_feature_names.items():
    print(model_key)
    print("Number of fitted features:", len(names))
    print("First 5:", names[:5])
    print()

open__logistic_regression__C_expanded
Number of fitted features: 60
First 5: ['numeric__age_clean', 'numeric__n_interests', 'numeric__subject_length', 'numeric__preheader_length', 'numeric__prior_email_count']

open__decision_tree__C_expanded
Number of fitted features: 60
First 5: ['numeric__age_clean', 'numeric__n_interests', 'numeric__subject_length', 'numeric__preheader_length', 'numeric__prior_email_count']

open__random_forest__C_expanded
Number of fitted features: 60
First 5: ['numeric__age_clean', 'numeric__n_interests', 'numeric__subject_length', 'numeric__preheader_length', 'numeric__prior_email_count']

click__logistic_regression__C_expanded
Number of fitted features: 60
First 5: ['numeric__age_clean', 'numeric__n_interests', 'numeric__subject_length', 'numeric__preheader_length', 'numeric__prior_email_count']

click__random_forest__C_expanded
Number of fitted features: 60
First 5: ['numeric__age_clean', 'numeric__n_interests', 'numeric__subject_length', 'numeric__preheader_l

## 9.16 Check class ordering and positive class index

In [64]:
class_ordering = {
    "open__logistic_regression__C_expanded": (
        open_logistic_pipeline.named_steps["model"].classes_.tolist()
    ),
    "open__decision_tree__C_expanded": (
        open_tree_pipeline.named_steps["model"].classes_.tolist()
    ),
    "open__random_forest__C_expanded": (
        open_rf_pipeline.named_steps["model"].classes_.tolist()
    ),
    "click__logistic_regression__C_expanded": (
        click_logistic_pipeline.named_steps["model"].classes_.tolist()
    ),
    "click__random_forest__C_expanded": (
        click_rf_pipeline.named_steps["model"].classes_.tolist()
    ),
}

positive_class_index = {}

for model_key, classes in class_ordering.items():
    print(model_key)
    print("Classes:", classes)
    
    assert 1.0 in classes, f"Class 1 missing for {model_key}"
    
    positive_class_index[model_key] = classes.index(1.0)
    
    print("Positive class index:", positive_class_index[model_key])
    print()

open__logistic_regression__C_expanded
Classes: [0.0, 1.0]
Positive class index: 1

open__decision_tree__C_expanded
Classes: [0.0, 1.0]
Positive class index: 1

open__random_forest__C_expanded
Classes: [0.0, 1.0]
Positive class index: 1

click__logistic_regression__C_expanded
Classes: [0.0, 1.0]
Positive class index: 1

click__random_forest__C_expanded
Classes: [0.0, 1.0]
Positive class index: 1



## 9.17 Compute trainval apparent sanity metrics only

Did the refitted models behave normally?

Did any model break?

Did probabilities become weird?

In [65]:
def compute_trainval_metrics(pipeline, X, y):

    model = pipeline.named_steps["model"]

    pos_idx = list(model.classes_).index(1.0)

    y_proba = pipeline.predict_proba(X)[:, pos_idx]
    y_pred = pipeline.predict(X)

    return {
        "roc_auc": roc_auc_score(y, y_proba),
        "pr_auc": average_precision_score(y, y_proba),
        "accuracy": accuracy_score(y, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "brier": brier_score_loss(y, y_proba),
        "log_loss": log_loss(y, y_proba),
        "predicted_positive_count": int(y_pred.sum()),
        "predicted_positive_rate": float(y_pred.mean()),
        "proba_min": float(y_proba.min()),
        "proba_max": float(y_proba.max()),
        "proba_mean": float(y_proba.mean()),
    }

In [66]:
# open logistic
open_logistic_metrics = compute_trainval_metrics(
    open_logistic_pipeline,
    X_open_logistic_trainval,
    y_open_logistic_trainval
)

open_logistic_metrics

{'roc_auc': 0.8503384281579355,
 'pr_auc': 0.8744545131309023,
 'accuracy': 0.773574938669218,
 'balanced_accuracy': 0.7640355911284993,
 'brier': 0.1546422578037381,
 'log_loss': 0.47441214210447474,
 'predicted_positive_count': 472057,
 'predicted_positive_rate': 0.603787799331309,
 'proba_min': 0.009646668299717881,
 'proba_max': 0.9803537427999222,
 'proba_mean': 0.5733189514279359}

In [67]:
# remaning models
open_tree_metrics = compute_trainval_metrics(
    open_tree_pipeline,
    X_open_tree_trainval,
    y_open_tree_trainval
)

open_rf_metrics = compute_trainval_metrics(
    open_rf_pipeline,
    X_open_rf_trainval,
    y_open_rf_trainval
)

click_logistic_metrics = compute_trainval_metrics(
    click_logistic_pipeline,
    X_click_logistic_trainval,
    y_click_logistic_trainval
)

click_rf_metrics = compute_trainval_metrics(
    click_rf_pipeline,
    X_click_rf_trainval,
    y_click_rf_trainval
)

print("Open Tree:")
print(open_tree_metrics)

print("\nOpen RF:")
print(open_rf_metrics)

print("\nClick Logistic:")
print(click_logistic_metrics)

print("\nClick RF:")
print(click_rf_metrics)

Open Tree:
{'roc_auc': 0.843003446250631, 'pr_auc': 0.844750421515243, 'accuracy': 0.7746595789856054, 'balanced_accuracy': 0.7632254176950442, 'brier': 0.1554746773704339, 'log_loss': 0.4777313058431889, 'predicted_positive_count': 482073, 'predicted_positive_rate': 0.6165988340116599, 'proba_min': 0.015472630347159016, 'proba_max': 0.9110644900108088, 'proba_mean': 0.5732733370340716}

Open RF:
{'roc_auc': 0.8690168944584995, 'pr_auc': 0.8925166204186602, 'accuracy': 0.7895618718231422, 'balanced_accuracy': 0.7824722659970409, 'brier': 0.1456087385964441, 'log_loss': 0.45130864429574735, 'predicted_positive_count': 461100, 'predicted_positive_rate': 0.5897731720357215, 'proba_min': 0.03402372561000282, 'proba_max': 0.9463852032311527, 'proba_mean': 0.5732351723238205}

Click Logistic:
{'roc_auc': 0.9661577872757968, 'pr_auc': 0.4789793695093934, 'accuracy': 0.9989610313053794, 'balanced_accuracy': 0.6284366753189904, 'brier': 0.000842782075301499, 'log_loss': 0.003999734777386435, 'p

## 9.18 Save fitted .joblib pipelines

In [68]:
# save fitted pipelines
model_paths = {
    "open__logistic_regression__C_expanded":
        OUTPUT_DIR / "9_open_refit_model_logistic_C_v1.joblib",

    "open__decision_tree__C_expanded":
        OUTPUT_DIR / "9_open_refit_model_decision_tree_C_v1.joblib",

    "open__random_forest__C_expanded":
        OUTPUT_DIR / "9_open_refit_model_random_forest_C_v1.joblib",

    "click__logistic_regression__C_expanded":
        OUTPUT_DIR / "9_click_refit_model_logistic_C_v1.joblib",

    "click__random_forest__C_expanded":
        OUTPUT_DIR / "9_click_refit_model_random_forest_C_v1.joblib",
}

In [70]:
# save models
joblib.dump(
    open_logistic_pipeline,
    model_paths["open__logistic_regression__C_expanded"],
    compress=3
)

joblib.dump(
    open_tree_pipeline,
    model_paths["open__decision_tree__C_expanded"],
    compress=3
)

joblib.dump(
    open_rf_pipeline,
    model_paths["open__random_forest__C_expanded"],
    compress=3
)

joblib.dump(
    click_logistic_pipeline,
    model_paths["click__logistic_regression__C_expanded"],
    compress=3
)

joblib.dump(
    click_rf_pipeline,
    model_paths["click__random_forest__C_expanded"],
    compress=3
)

['/Users/khanhhien/Documents/Thesis/Outputs/9_click_refit_model_random_forest_C_v1.joblib']

In [71]:
# verify
for model_name, model_path in model_paths.items():

    exists = model_path.exists()

    print(
        model_name,
        "->",
        exists
    )

    assert exists

open__logistic_regression__C_expanded -> True
open__decision_tree__C_expanded -> True
open__random_forest__C_expanded -> True
click__logistic_regression__C_expanded -> True
click__random_forest__C_expanded -> True


## 9.19 Save summary CSV

In [72]:
refit_summary_rows = [
    {
        "target": "open",
        "candidate_key": "logistic_regression__C_expanded",
        "model_type": "logistic_regression",
        "n_raw_features": len(open_feature_set_C_expanded),
        "n_fitted_features": len(fitted_feature_names["open__logistic_regression__C_expanded"]),
        "fit_runtime_seconds": open_logistic_fit_runtime_seconds,
        "classes": class_ordering["open__logistic_regression__C_expanded"],
        "positive_class_index": positive_class_index["open__logistic_regression__C_expanded"],
        **{f"trainval_apparent_{k}": v for k, v in open_logistic_metrics.items()},
        "model_path": str(model_paths["open__logistic_regression__C_expanded"]),
    },
    {
        "target": "open",
        "candidate_key": "decision_tree__C_expanded",
        "model_type": "decision_tree",
        "n_raw_features": len(open_feature_set_C_expanded),
        "n_fitted_features": len(fitted_feature_names["open__decision_tree__C_expanded"]),
        "fit_runtime_seconds": open_tree_fit_runtime_seconds,
        "classes": class_ordering["open__decision_tree__C_expanded"],
        "positive_class_index": positive_class_index["open__decision_tree__C_expanded"],
        **{f"trainval_apparent_{k}": v for k, v in open_tree_metrics.items()},
        "model_path": str(model_paths["open__decision_tree__C_expanded"]),
    },
    {
        "target": "open",
        "candidate_key": "random_forest__C_expanded",
        "model_type": "random_forest",
        "n_raw_features": len(open_feature_set_C_expanded),
        "n_fitted_features": len(fitted_feature_names["open__random_forest__C_expanded"]),
        "fit_runtime_seconds": open_rf_fit_runtime_seconds,
        "classes": class_ordering["open__random_forest__C_expanded"],
        "positive_class_index": positive_class_index["open__random_forest__C_expanded"],
        **{f"trainval_apparent_{k}": v for k, v in open_rf_metrics.items()},
        "model_path": str(model_paths["open__random_forest__C_expanded"]),
    },
    {
        "target": "click",
        "candidate_key": "logistic_regression__C_expanded",
        "model_type": "logistic_regression",
        "n_raw_features": len(click_feature_set_C_expanded),
        "n_fitted_features": len(fitted_feature_names["click__logistic_regression__C_expanded"]),
        "fit_runtime_seconds": click_logistic_fit_runtime_seconds,
        "classes": class_ordering["click__logistic_regression__C_expanded"],
        "positive_class_index": positive_class_index["click__logistic_regression__C_expanded"],
        **{f"trainval_apparent_{k}": v for k, v in click_logistic_metrics.items()},
        "model_path": str(model_paths["click__logistic_regression__C_expanded"]),
    },
    {
        "target": "click",
        "candidate_key": "random_forest__C_expanded",
        "model_type": "random_forest",
        "n_raw_features": len(click_feature_set_C_expanded),
        "n_fitted_features": len(fitted_feature_names["click__random_forest__C_expanded"]),
        "fit_runtime_seconds": click_rf_fit_runtime_seconds,
        "classes": class_ordering["click__random_forest__C_expanded"],
        "positive_class_index": positive_class_index["click__random_forest__C_expanded"],
        **{f"trainval_apparent_{k}": v for k, v in click_rf_metrics.items()},
        "model_path": str(model_paths["click__random_forest__C_expanded"]),
    },
]

refit_summary_df = pd.DataFrame(refit_summary_rows)

REFIT_SUMMARY_PATH = OUTPUT_DIR / "9_refit_summary_v1.csv"

refit_summary_df.to_csv(
    REFIT_SUMMARY_PATH,
    index=False
)

print("Saved refit summary to:")
print(REFIT_SUMMARY_PATH)

refit_summary_df

Saved refit summary to:
/Users/khanhhien/Documents/Thesis/Outputs/9_refit_summary_v1.csv


,target,candidate_key,model_type,n_raw_features,n_fitted_features,fit_runtime_seconds,classes,positive_class_index,trainval_apparent_roc_auc,trainval_apparent_pr_auc,trainval_apparent_accuracy,trainval_apparent_balanced_accuracy,trainval_apparent_brier,trainval_apparent_log_loss,trainval_apparent_predicted_positive_count,trainval_apparent_predicted_positive_rate,trainval_apparent_proba_min,trainval_apparent_proba_max,trainval_apparent_proba_mean,model_path
0,open,logistic_regression__C_expanded,logistic_regression,28,60,4.999010,"[0.0, 1.0]",1,0.850338,0.874455,0.773575,0.764036,0.154642,0.474412,472057,0.603788,0.009647,0.980354,0.573319,/Users/khanhhien/Documents/Thesis/Outputs/9_op...
1,open,decision_tree__C_expanded,decision_tree,28,60,3.017949,"[0.0, 1.0]",1,0.843003,0.844750,0.774660,0.763225,0.155475,0.477731,482073,0.616599,0.015473,0.911064,0.573273,/Users/khanhhien/Documents/Thesis/Outputs/9_op...
2,open,random_forest__C_expanded,random_forest,28,60,37.004086,"[0.0, 1.0]",1,0.869017,0.892517,0.789562,0.782472,0.145609,0.451309,461100,0.589773,0.034024,0.946385,0.573235,/Users/khanhhien/Documents/Thesis/Outputs/9_op...
3,click,logistic_regression__C_expanded,logistic_regression,28,60,4.846308,"[0.0, 1.0]",1,0.966158,0.478979,0.998961,0.628437,0.000843,0.004000,360,0.000442,0.000007,0.947750,0.001304,/Users/khanhhien/Documents/Thesis/Outputs/9_cl...
4,click,random_forest__C_expanded,random_forest,28,60,131.297979,"[0.0, 1.0]",1,0.998717,0.834655,0.999399,0.762491,0.000476,0.002016,539,0.000662,0.000008,1.000000,0.001223,/Users/khanhhien/Documents/Thesis/Outputs/9_cl...


## 9.20 Save metadata JSON

In [73]:
refit_metadata = {
    "project": "B2C email engagement prediction thesis",
    "stage": "Stage 9 - Refit selected tuned models",
    "notebook": "9_refit_selected_models_v1.ipynb",
    "random_state": RANDOM_STATE,
    "n_jobs": N_JOBS,
    "group_col": GROUP_COL,
    "open_target": OPEN_TARGET,
    "click_target": CLICK_TARGET,
    "split_logic": "Earliest 80% campaigns used as train-validation; latest 20% campaigns held out as final chronological test.",
    "final_test_status": "Recreated for split checks only; no final test predictions or metrics computed in Stage 9.",
    "open_split": {
        "trainval_rows": int(open_trainval_df.shape[0]),
        "test_rows": int(open_test_df.shape[0]),
        "trainval_campaigns": int(open_trainval_df[GROUP_COL].nunique()),
        "test_campaigns": int(open_test_df[GROUP_COL].nunique()),
        "trainval_campaign_min": int(open_trainval_df[GROUP_COL].min()),
        "trainval_campaign_max": int(open_trainval_df[GROUP_COL].max()),
        "test_campaign_min": int(open_test_df[GROUP_COL].min()),
        "test_campaign_max": int(open_test_df[GROUP_COL].max()),
    },
    "click_split": {
        "trainval_rows": int(click_trainval_df.shape[0]),
        "test_rows": int(click_test_df.shape[0]),
        "trainval_campaigns": int(click_trainval_df[GROUP_COL].nunique()),
        "test_campaigns": int(click_test_df[GROUP_COL].nunique()),
        "trainval_campaign_min": int(click_trainval_df[GROUP_COL].min()),
        "trainval_campaign_max": int(click_trainval_df[GROUP_COL].max()),
        "test_campaign_min": int(click_test_df[GROUP_COL].min()),
        "test_campaign_max": int(click_test_df[GROUP_COL].max()),
    },
    "feature_sets": {
        "open_C_expanded": open_feature_set_C_expanded,
        "click_C_expanded": click_feature_set_C_expanded,
    },
    "feature_types": {
        "open_C": open_feature_types_C,
        "click_C": click_feature_types_C,
    },
    "carry_forward_params": {
        "open": open_carry_forward_params,
        "click": click_carry_forward_params,
    },
    "model_paths": {
        key: str(path)
        for key, path in model_paths.items()
    },
    "summary_file": str(REFIT_SUMMARY_PATH),
    "notes": [
        "All final selected models were refitted on the full train-validation set.",
        "Trainval apparent metrics are sanity checks only and should not be interpreted as final performance.",
        "Final chronological test metrics must be computed only in Stage 10."
    ],
}

REFIT_METADATA_PATH = OUTPUT_DIR / "9_refit_metadata_v1.json"

with open(REFIT_METADATA_PATH, "w") as f:
    json.dump(refit_metadata, f, indent=4)

assert REFIT_METADATA_PATH.exists()

print("Saved metadata to:")
print(REFIT_METADATA_PATH)

Saved metadata to:
/Users/khanhhien/Documents/Thesis/Outputs/9_refit_metadata_v1.json


## 9.21 Save fitted feature names JSON

In [74]:
FITTED_FEATURE_NAMES_PATH = (
    OUTPUT_DIR / "9_refit_fitted_feature_names_v1.json"
)

with open(
    FITTED_FEATURE_NAMES_PATH,
    "w"
) as f:

    json.dump(
        fitted_feature_names,
        f,
        indent=4
    )

assert FITTED_FEATURE_NAMES_PATH.exists()

print("Saved fitted feature names to:")
print(FITTED_FEATURE_NAMES_PATH)

Saved fitted feature names to:
/Users/khanhhien/Documents/Thesis/Outputs/9_refit_fitted_feature_names_v1.json


## 9.22 Save package versions JSON

In [75]:
package_versions = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
}

PACKAGE_VERSIONS_PATH = (
    OUTPUT_DIR / "9_refit_package_versions_v1.json"
)

with open(
    PACKAGE_VERSIONS_PATH,
    "w"
) as f:

    json.dump(
        package_versions,
        f,
        indent=4
    )

assert PACKAGE_VERSIONS_PATH.exists()

print("Saved package versions to:")
print(PACKAGE_VERSIONS_PATH)

Saved package versions to:
/Users/khanhhien/Documents/Thesis/Outputs/9_refit_package_versions_v1.json


## 9.23 Reload saved artifacts to verify

In [76]:
# reload saved models
reloaded_models = {}

for model_key, model_path in model_paths.items():

    reloaded_models[model_key] = joblib.load(model_path)

    print(
        model_key,
        "loaded successfully"
    )

open__logistic_regression__C_expanded loaded successfully
open__decision_tree__C_expanded loaded successfully
open__random_forest__C_expanded loaded successfully
click__logistic_regression__C_expanded loaded successfully
click__random_forest__C_expanded loaded successfully


In [78]:
# verify pipeline structure
for model_key, model in reloaded_models.items():

    print("\n", model_key)

    print(
        "Has preprocessor:",
        "preprocessor" in model.named_steps
    )

    print(
        "Has model:",
        "model" in model.named_steps
    )

    assert "preprocessor" in model.named_steps
    assert "model" in model.named_steps


 open__logistic_regression__C_expanded
Has preprocessor: True
Has model: True

 open__decision_tree__C_expanded
Has preprocessor: True
Has model: True

 open__random_forest__C_expanded
Has preprocessor: True
Has model: True

 click__logistic_regression__C_expanded
Has preprocessor: True
Has model: True

 click__random_forest__C_expanded
Has preprocessor: True
Has model: True


In [79]:
# prediction smoke test
reload_checks = [
    (
        reloaded_models["open__logistic_regression__C_expanded"],
        X_open_logistic_trainval.head(5)
    ),
    (
        reloaded_models["open__decision_tree__C_expanded"],
        X_open_tree_trainval.head(5)
    ),
    (
        reloaded_models["open__random_forest__C_expanded"],
        X_open_rf_trainval.head(5)
    ),
    (
        reloaded_models["click__logistic_regression__C_expanded"],
        X_click_logistic_trainval.head(5)
    ),
    (
        reloaded_models["click__random_forest__C_expanded"],
        X_click_rf_trainval.head(5)
    ),
]

for model, sample_X in reload_checks:

    proba = model.predict_proba(sample_X)

    print(
        type(model.named_steps["model"]).__name__,
        "->",
        proba.shape
    )

LogisticRegression -> (5, 2)
DecisionTreeClassifier -> (5, 2)
RandomForestClassifier -> (5, 2)
LogisticRegression -> (5, 2)
RandomForestClassifier -> (5, 2)


## 9.24 Assert no final test predictions/metrics exist

In [80]:
print("Final test datasets exist:")

print("Open test rows:", len(open_test_df))
print("Click test rows:", len(click_test_df))

print("\nStage 9 policy:")

print(
    "No final test predictions were generated."
)

print(
    "No final test metrics were computed."
)

print(
    "Final chronological test sets remain untouched."
)

Final test datasets exist:
Open test rows: 275718
Click test rows: 289973

Stage 9 policy:
No final test predictions were generated.
No final test metrics were computed.
Final chronological test sets remain untouched.


## 9.25 Save completion note

In [81]:
completion_note = """
Stage 9 completed successfully.

Actions completed:
- Recreated chronological train-validation and final test split.
- Loaded carry-forward parameters.
- Refit five selected final models:
    * Open Logistic Regression C
    * Open Decision Tree C
    * Open Random Forest C
    * Click Logistic Regression C
    * Click Random Forest C
- Extracted fitted feature names.
- Verified class ordering.
- Computed train-validation apparent sanity metrics.
- Saved fitted pipelines.
- Saved summary CSV.
- Saved metadata JSON.
- Saved fitted feature names JSON.
- Saved package versions JSON.
- Reloaded all saved pipelines successfully.

Important:
- No final test predictions were generated.
- No final test metrics were computed.
- Final chronological test evaluation is reserved for Stage 10.
"""

COMPLETION_NOTE_PATH = (
    OUTPUT_DIR / "9_refit_completion_note_v1.txt"
)

with open(
    COMPLETION_NOTE_PATH,
    "w"
) as f:

    f.write(completion_note)

assert COMPLETION_NOTE_PATH.exists()

print("Stage 9 completed.")
print("Completion note saved to:")
print(COMPLETION_NOTE_PATH)

Stage 9 completed.
Completion note saved to:
/Users/khanhhien/Documents/Thesis/Outputs/9_refit_completion_note_v1.txt
